# Tensorflow Input Pipeline


In [1]:
import tensorflow as tf

In [2]:
daily_sales_numbers = [21, 22, -108, 31, -1, 32, 34, 31]

In [3]:
tf_dataset = tf.data.Dataset.from_tensor_slices(daily_sales_numbers)
tf_dataset

<_TensorSliceDataset element_spec=TensorSpec(shape=(), dtype=tf.int32, name=None)>

In [4]:
for sales in tf_dataset.as_numpy_iterator():
    print(sales)

21
22
-108
31
-1
32
34
31


In [5]:
for sales in tf_dataset.take(3).as_numpy_iterator():
    print(sales)

21
22
-108


In [6]:
tf_dataset = tf_dataset.filter(lambda x: x > 0)
for sales in tf_dataset.as_numpy_iterator():
    print(sales)

21
22
31
32
34
31


In [7]:
tf_dataset = tf_dataset.map(lambda x: x * 72)
for sales in tf_dataset.as_numpy_iterator():
    print(sales)

1512
1584
2232
2304
2448
2232


In [8]:
tf_dataset = tf_dataset.shuffle(3)
for sales in tf_dataset.as_numpy_iterator():
    print(sales)

1512
2304
2448
1584
2232
2232


In [9]:
for sales_batch in tf_dataset.batch(4).as_numpy_iterator():
    print(sales_batch)

[1584 2304 2448 2232]
[1512 2232]


In [10]:
tf_dataset = tf.data.Dataset.from_tensor_slices(daily_sales_numbers)
tf_dataset = (
    tf_dataset.filter(lambda x: x > 0).map(lambda y: y * 72).shuffle(2).batch(2)
)
for sales_batch in tf_dataset.as_numpy_iterator():
    print(sales_batch)

[1512 1584]
[2304 2448]
[2232 2232]


In [12]:
images_ds = tf.data.Dataset.list_files("./images/*/*", shuffle=False)

for file in images_ds.take(5):
    print(file.numpy())

b'.\\images\\cat\\20 Reasons Why Cats Make the Best Pets....jpg'
b'.\\images\\cat\\7 Foods Your Cat Can_t Eat.jpg'
b'.\\images\\cat\\A cat appears to have caught the....jpg'
b'.\\images\\cat\\Adopt-A-Cat Month\xc2\xae - American Humane....jpg'
b'.\\images\\cat\\All About Your Cat_s Tongue.jpg'


In [27]:
images_ds = images_ds.shuffle(200)
for file in images_ds.take(5):
    print(file.numpy())

b'.\\images\\cat\\Is My Cat Normal_.jpg'
b'.\\images\\dog\\9 Reasons to Own a Dog.jpg'
b'.\\images\\dog\\What makes dogs so special and....jpg'
b'.\\images\\dog\\Service Dogs from Southeastern Guide Dogs.jpg'
b'.\\images\\dog\\How to make your dog feel comfortable....jpg'


In [28]:
class_names = ["cat", "dog"]

In [30]:
image_count = len(images_ds)
print(image_count)

130


In [31]:
train_size = int(image_count * 0.8)


train_ds = images_ds.take(train_size)
test_ds = images_ds.skip(train_size)

In [32]:
len(train_ds), len(test_ds)

(104, 26)

In [33]:
s = ".\\images\\cat\\Is My Cat Normal_.jpg"
s.split("\\")[-2]

'cat'

In [39]:
import os


def get_label(file_path: str) -> str:
    return tf.strings.split(file_path, os.sep)[-2]

In [42]:
def process_image(file_path):
    label = get_label(file_path)
    image = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(image)
    img = tf.image.resize(img, [128, 128])
    return img, label

In [40]:
for t in train_ds.take(5):
    print(t.numpy())

b'.\\images\\cat\\Cat Throwing Up_ Normal or Cause for....jpg'
b'.\\images\\cat\\Are Cats Domesticated_ _ The New Yorker.jpg'
b'.\\images\\dog\\Canine Mind....jpg'
b'.\\images\\dog\\Calculate Your Dog_s Age With This New....jpg'
b'.\\images\\dog\\Haunted Victorian Child_ Dog....jpg'


In [44]:
for img, label in train_ds.map(process_image).take(5):
    print(img.numpy())
    print(label.numpy())

[[[232.      190.      191.     ]
  [232.      190.      191.     ]
  [232.      190.      191.     ]
  ...
  [234.      192.      193.     ]
  [235.      193.      194.     ]
  [235.      193.      194.     ]]

 [[233.15625 191.15625 192.15625]
  [232.      190.      191.     ]
  [233.      191.      192.     ]
  ...
  [235.      193.      194.     ]
  [235.      193.      194.     ]
  [234.84375 192.84375 193.84375]]

 [[232.      190.      191.     ]
  [232.      190.      191.     ]
  [233.      191.      192.     ]
  ...
  [235.      193.      194.     ]
  [235.      193.      194.     ]
  [235.      193.      194.     ]]

 ...

 [[233.40625 193.40625 193.40625]
  [235.      193.      194.     ]
  [234.      194.      194.     ]
  ...
  [234.      192.      193.     ]
  [234.      192.      193.     ]
  [234.      192.      193.     ]]

 [[235.      195.      195.     ]
  [235.      193.      194.     ]
  [234.      194.      194.     ]
  ...
  [235.      193.      194.     ]
  [2

In [45]:
def scale_image(image, label):
    image = image / 255.0
    return image, label

In [46]:
train_ds = train_ds.map(process_image).map(scale_image)

In [50]:
for img, label in train_ds.take(5):
    print("***Image***", img.numpy()[0][0])
    print("***Label***", label.numpy())

***Image*** [0.60398287 0.6628064  0.6510417 ]
***Label*** b'dog'
***Image*** [0.1171492  0.15121017 0.16689645]
***Label*** b'cat'
***Image*** [0.38039216 0.16470589 0.52156866]
***Label*** b'cat'
***Image*** [0.23137255 0.2784314  0.33333334]
***Label*** b'dog'
***Image*** [0.9059021  0.9137452  0.85884327]
***Label*** b'dog'
